# Tutorial: Is This Catalog Any Good? First steps to qualifying LSST DP2.

Contact author: Alex Broughton
<br>Date: Wednesday, October 7, 2026

In [ ]:
! eups list -s | grep lsst_distrib

## Introduction

Someone hands you a catalog of nearly 800 million objects. Before you do any science with it, you need to know what you're holding: where the data are, how deep they go, which pixels you can trust, and what the pipeline did to turn images into rows of numbers. We are going to learn a few of the first steps one might do to qualify the data in a survey for their science.

## 1.0 Set Up

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval
import astropy.units as u
from astropy.coordinates import SkyCoord
from scipy import ndimage

from lsst.daf.butler import Butler
from lsst.images import Box

%matplotlib inline
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 12,
    "axes.labelsize": 12,
    "legend.fontsize": 12,
    "figure.dpi": 120,
})

In [ ]:
butler = Butler("dp2", collections="dp2")

In [ ]:
def plot_coadd(exposure):
    """Coadd image on R.A./Dec. axes; returns the axes and the image extent in pixels."""
    bbox = exposure.bbox
    extent = (bbox.x.start - 0.5, bbox.x.stop - 0.5, bbox.y.start - 0.5, bbox.y.stop - 0.5)
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(projection=exposure.astropy_wcs)
    
    # norm = ImageNormalize(exposure.image.array, interval=ZScaleInterval(), stretch=AsinhStretch())

    vmin, _ = ZScaleInterval().get_limits(exposure.image.array)
    vmax = np.nanpercentile(exposure.image.array, 99.9)
    norm = ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch(a=0.05))
    
    ax.imshow(exposure.image.array, origin="lower", extent=extent, cmap="gray", norm=norm)
    ax.autoscale(False)
    ax.coords[0].set_axislabel(r"$\mathrm{R.A.}$ [deg]")
    ax.coords[1].set_axislabel(r"$\mathrm{Dec.}$ [deg]")
    ax.coords[0].set_major_formatter("d.dd")
    ax.coords[1].set_major_formatter("d.dd")
    return ax, extent

## 2.0 Survey Property Maps

The LSST Data Reduction Pipelines take the raw data, which is often sparse and non-uniform, and aggregates it into a set of calibrated, masked, combined, and verified data products. The final level of data products are the `deep_coadds` and `object` catalogs. 

The input data that goes into the pipeline gets summarized by the **survey property maps**: for every band, the total exposure time, the PSF size, the sky noise, a PSF magnitude limit (a depth estimate), and more, each stored on the sky on a common sky partition and coordinate system called a HEALPix map. They are the perfect place to start to qualify a dataset when it is handed to you.

This section (roughly) follows DP2 tutorial [203.1](https://dp2.lsst.io/tutorials/notebook/203/notebook-203-1.html).

### HEALPix 
HEALPix divides the sky into equal-area pixels, and `nside` sets how many: a larger `nside` means smaller pixels. DP2's maps are stored at `nside = 32768` (6.4″ pixels). The full-resolution maps are large (too large to fit the the 32GB notebook you have open). Luckily, LSST has built a smart middleware tool to take care of this problem. The Butler is basically a smarter I/O module that prevents you from ever having to open a massive FITS file; it is fully aware of where the data lives (file paths), but also properties of that data (dimensions like tract, band, exposure ID, etc.), and it able to grab only the specific pieces of data that you want without ever having to open a massive data file. For survey property maps, the Butler can  **degrade** a map while it reads it to a specified `nside`.

Start by choosing a sky position and band. You can scan through the LSST Data Preview 2 (DP2) HiPS maps [here](https://data.lsst.cloud/portal/app/) to grab image coordinates for an interesting part of the sky. Everything from here on uses this position and band.

In [ ]:
import hpgeom as hpg
import pandas as pd
import skyproj

ra, dec, band = 187.2645519, 8.6612496, "i"  # EQ_J2000

SKYMAP = "lsst_cells_v2"
NSIDE_COV = 32         # resolution of the coverage map
NSIDE_NATIVE = 32768   # native resolution of the maps
PIXEL_SCALE = 0.2      # arcsec per coadd pixel
SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))

Each property map is its own Butler dataset type, with the statistic in the name (`sum`, `weighted_mean`, ...). List them:

In [ ]:
for dtype in sorted(butler.registry.queryDatasetTypes(expression="*consolidated_map*")):
    print(dtype.name)

This notebook uses four of them. The PSF size is stored as a width $\sigma$ in pixels; `scale` converts it to a FWHM in arcseconds so it can be compared with seeing.

In [ ]:
MAPS = {
    "psf_maglim": dict(
        dtype="deepCoadd_psf_maglim_consolidated_map_weighted_mean",
        scale=1.0, label="PSF magnitude limit [AB mag]"),
    "psf_fwhm": dict(
        dtype="deepCoadd_psf_size_consolidated_map_weighted_mean",  # sigma, in pixels
        scale=SIGMA_TO_FWHM * PIXEL_SCALE, label="PSF FWHM [arcsec]"),
    "exposure_time": dict(
        dtype="deepCoadd_exposure_time_consolidated_map_sum",
        scale=1.0, label="Exposure time [s]"),
    "sky_noise": dict(
        dtype="deepCoadd_sky_noise_consolidated_map_weighted_mean",
        scale=1.0, label="Sky noise [nJy]"),
}

### How big is a map pixel?

A HEALPix pixel has an area of $4\pi / (12\, n_\mathrm{side}^2)$ steradians (there are $12\, n_\mathrm{side}^2$ pixels on the sky). Here is the side of a square of that area, next to the 0.2″ pixels of the coadd images.

In [ ]:
def pixel_size_arcsec(nside):
    """Side length of a square with the same area as one HEALPix pixel, in arcsec."""
    return np.degrees(np.sqrt(4 * np.pi / (12 * nside**2))) * 3600

nsides = [32, 512, 4096, 32768]
pd.DataFrame({
    "nside": nsides,
    "pixel size [arcsec]": [round(pixel_size_arcsec(n), 1) for n in nsides],
    "coadd pixels across": [round(pixel_size_arcsec(n) / PIXEL_SCALE) for n in nsides],
})

### Reading a map

`load_map` reads one map for the chosen band. The Butler takes the list of coverage `pixels` to read and, optionally, a `degrade_nside`; reading everywhere the map has data at low resolution is cheap, but a full-resolution read of the whole footprint is not. `sample` looks up the map at sky positions. 

In [ ]:
def load_map(key, nside=None, pixels=None):
    """Read one survey property map for `band`.

    nside:  degrade to this resolution while reading (None = native).
    pixels: coverage pixels to read (None = everywhere the map has data).
    """
    dtype = MAPS[key]["dtype"]
    if pixels is None:
        coverage = butler.get(f"{dtype}.coverage", band=band, skymap=SKYMAP)
        pixels, = np.where(coverage.coverage_mask)
    parameters = {"pixels": [int(p) for p in pixels]}
    if nside is not None and nside < NSIDE_NATIVE:
        parameters["degrade_nside"] = nside
    return butler.get(dtype, band=band, skymap=SKYMAP, parameters=parameters)


def sample(hsp_map, ra_deg, dec_deg):
    """Map values at sky positions (degrees), NaN where the map has no data."""
    values = hsp_map.get_values_pos(np.asarray(ra_deg, float), np.asarray(dec_deg, float), lonlat=True)
    values = np.asarray(values, dtype=float)
    return np.where(values < -1e29, np.nan, values)


_maps = {}

def get_map(key, nside):
    """Cached read of a map, restricted to the coverage pixels within 0.5 deg of the target position."""
    cache_key = (ra, dec, band, key, nside)
    if cache_key not in _maps:
        coverage = butler.get(f"{MAPS[key]['dtype']}.coverage", band=band, skymap=SKYMAP)
        near = hpg.query_circle(NSIDE_COV, ra, dec, 0.5, inclusive=True)
        near = [p for p in near if coverage.coverage_mask[p]]
        _maps[cache_key] = load_map(key, nside, pixels=near)
    return _maps[cache_key]

### Where is the data?

First the whole DP2 footprint. At `nside = 512` (412″ pixels) reading every coverage pixel is quick, and that is plenty to see where the data are. The star is the position chosen above.

In [ ]:
footprint = load_map("psf_maglim", nside=512)

fig_fp, ax_fp = plt.subplots(figsize=(9, 5))
sp = skyproj.McBrydeSkyproj(ax=ax_fp, lon_0=0.0)
sp.draw_hspmap(footprint)
sp.draw_colorbar(label=MAPS["psf_maglim"]["label"])
sp.ax.scatter([ra], [dec], marker="*", s=200, c="tab:red", zorder=5)
plt.show()

**QUESTION:** Why does the map look like this? What is the significance of the large band we call the Science Validation (SV) region? 

Note that this is only the "coadded" region. We have data from most other parts of the sky, but in other places it is only 1-2 exposures deep.

### Degrading a map

Now zoom in on a 0.5° box around the position and compare the same map at three resolutions: the native `nside = 32768`, the `nside = 4096` that tutorial 203.1 recommends, and a coarse `nside = 512`. Only the coverage pixels near the target are read, so even the native read is small.

In [ ]:
half = 0.25  # deg, half-width of the box
RA, DEC = np.meshgrid(
    ra + np.linspace(-half, half, 251) / np.cos(np.deg2rad(dec)),
    dec + np.linspace(-half, half, 251),
)

levels = (NSIDE_NATIVE, 4096, 512)
grids = {n: sample(get_map("psf_maglim", n), RA, DEC) for n in levels}
vmin, vmax = np.nanpercentile(grids[NSIDE_NATIVE], [1, 99])

fig_deg, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharex=True, sharey=True, constrained_layout=True)
for ax_, n in zip(axes, levels):
    pcm = ax_.pcolormesh(RA, DEC, grids[n], vmin=vmin, vmax=vmax, cmap="viridis", shading="auto")
    ax_.plot(ra, dec, "r*", ms=12)
    ax_.set_title(rf"$n_\mathrm{{side}} = {n}$ ({pixel_size_arcsec(n):.0f} arcsec pixels)")
    ax_.set_xlabel(r"$\mathrm{R.A.}$ [deg]")
axes[0].set_ylabel(r"$\mathrm{Dec.}$ [deg]")
axes[0].invert_xaxis()  # East to the left; the axes are shared
fig_deg.colorbar(pcm, ax=axes, label=MAPS["psf_maglim"]["label"])
plt.show()

Finally, the four maps at the chosen position, at each resolution. (This also reads every map near the position once, so the later sections can reuse them.)

In [ ]:
rows = {}
for key, spec in MAPS.items():
    rows[spec["label"]] = {
        f"nside {n}": float(sample(get_map(key, n), ra, dec)) * spec["scale"] for n in levels
    }
pd.DataFrame(rows).T.round(3)

**QUESTION:** A map pixel summarizes dozens of input images. What does that summary hide?

- The PSF size map is a *weighted mean of the input visits' PSF widths*. Each coadd pixel is a local linear average of the true (static) sky as long as the weights do not depend on the scene, but the averaging scheme changes from pixel to pixel.

- Ideally, we want the coadd to have a well-defined PSF. Imagine averaging two normalized Gaussian PSFs, what happens if they have different sizes; what happens if they have different amounts of sky noise?. Are the resulting coadded PSFs still a well-defined by a Gaussian? What information do you lose between measuring a source in a single image and a coadd image in each case?

- Some experiments (KiDS, Euclid etc.) build median coadds for detection and revert to single visit images for measurement. This is too expensive to do for LSST where the number of images at any point are order of magnitudes more. What is the optimal weighting scheme for LSST (equal weights? inverse-variance weight?)? 

- When does the optimal weight break down? Is there the same optimal weight everywhere?

## 3.0 Load the Image and Catalog

Load the coadd patch that contains the position chosen in section 2.0, and the object catalog for its tract.

In [ ]:
# `ra`, `dec` and `band` were set in section 2.0

# Coadd patch overlapping this sky position
query = "band.name = :band AND patch.region OVERLAPS POINT(:ra, :dec)"
bind = {"band": band, "ra": ra, "dec": dec}
refs = butler.query_datasets(
    "deep_coadd",  # DATASET TYPE
    where=query,   # SEARCH
    bind=bind,     # METADATA CONSTRAINTS
    order_by="patch", # SORT THE OUTPUT
)

deep_coadd = butler.get(refs[0])

# This is a stylistic plotting choice, see `https://github.com/lsst/dp1_lsst_io/blob/main/docs/tutorials/notebook/202/202_1_Deep_coadds.html` for more details.
deep_coadd.apply_background("pretty")  #  try also, "object"
dataId = refs[0].dataId

In [ ]:
# Object catalog for the tract, trimmed to a few columns, then a S/N cut within this patch

columns = ["objectId", "patch", "parentObjectId", "detect_fromBlend", "detect_isIsolated", f"{band}_ra", f"{band}_dec", f"{band}_psfFlux", f"{band}_psfFluxErr", f"{band}_sersicFlux", f"{band}_sersicFluxErr"]
objects = butler.get("object", tract=dataId["tract"], parameters={"columns": columns})

sn_threshold = 300
# S/N cut on the Sersic (model) flux; swap in the PSF flux for a stellar sample
high_sn = objects[f"{band}_sersicFlux"] / objects[f"{band}_sersicFluxErr"] > sn_threshold
in_patch = objects["patch"] == dataId["patch"]
objects = objects[high_sn & in_patch]

# Pixel positions on the coadd
objects_xy = deep_coadd.sky_projection.sky_to_pixel(
    SkyCoord(objects[f"{band}_ra"] * u.deg, objects[f"{band}_dec"] * u.deg)
)
len(objects)

**QUESTION:** What *is* one row of the `object` table? Each row holds forced measurements in all six bands at the position of a source detected in a visit image or a deep coadd at S/N ≥ 5, and only deblended children are kept ([tutorial 201.01](https://dp2.lsst.io/tutorials/notebook/201/notebook-201-01.html)).

- Pick one object above and write down, in order, every step between photons hitting the CCD and that row. Which step could fail without anyone noticing?
- "Forced" means a flux is measured whether or not anything is there. How many objects in this patch have a *negative* PSF flux (the first cell of section 8.0 counts the positive ones)? How can a flux be negative? If you drop those objects, what happens to the average flux of a faint sample?
- One band (`refBand`) sets each object's position and shape for the forced measurements in every other band. When could that choice hurt you?

## 4.0 Overlay Catalog on Image

In [ ]:
ax, extent = plot_coadd(deep_coadd)
plt.show()

In [ ]:
ax, extent = plot_coadd(deep_coadd)
ax.scatter(objects_xy.x, objects_xy.y, s=60, facecolors="none", edgecolors="tab:orange", linewidths=0.8)
ax.legend(
    handles=[Line2D([], [], marker="o", ls="", mfc="none", mec="tab:orange", label=r"Sérsic $\mathrm{S/N} > 300$ objects")],
    loc="upper right",
)
plt.show()

## 5.0 Add Detection Footprints

In [ ]:
ax, extent = plot_coadd(deep_coadd)
ax.scatter(objects_xy.x, objects_xy.y, s=60, facecolors="none", edgecolors="tab:orange", linewidths=0.8)

# DETECTED marks pixels belonging to a source footprint on the coadd
detected = deep_coadd.mask.get("DETECTED")
# ax.contour(detected, levels=[0.5], extent=extent, origin="lower", colors="tab:green", linewidths=0.6)

ax.imshow(
    np.ma.masked_where(~detected, detected), origin="lower", extent=extent,
    cmap=ListedColormap(["#009E73"]), alpha=0.5,
)
ax.contour(detected, levels=[0.5], extent=extent, origin="lower", colors="#009E73", linewidths=0.6)

ax.legend(
    handles=[
        Line2D([], [], marker="o", ls="", mfc="none", mec="tab:orange", label=r"Sérsic $\mathrm{S/N} > 300$ objects"),
        Patch(facecolor="#009E73", alpha=0.5, label="Detection footprints"),
    ],
    loc="upper right",
)
plt.show()

**QUESTION:** What fraction of objects are blended? HSC found that 58% of objects in its Wide layer are blended ([Bosch et al. 2018](https://arxiv.org/abs/1705.06766)). 

In [ ]:
# Blended objects in this patch: objects deblended from a parent footprint that held more than one object
#
# Columns (DP2 Object table):
#   objectId           unique ID of each object (one row per deblended object)
#   patch              skymap patch the object belongs to
#   parentObjectId     ID of the parent: the original detection footprint this object was deblended from.
#                      Objects split from the same footprint share a parentObjectId. Parents are not rows
#                      in the Object table, which keeps only deblended children.
#   detect_fromBlend   True if the object was deblended from a parent with more than one child
#   detect_isIsolated  True if the object is not part of a blend (it was the only thing in its footprint)

# Code here

In [ ]:
from_blend = np.asarray(objects["detect_fromBlend"], dtype=bool)
isolated = np.asarray(objects["detect_isIsolated"], dtype=bool)
parents = np.asarray(objects["parentObjectId"])

n_obj = len(objects)
n_blended = from_blend.sum()
n_groups = len(np.unique(parents[from_blend]))

print(f"{n_blended} of {n_obj} objects ({n_blended / n_obj:.0%}) are blended")
print(f"from {n_groups} blended footprints, {n_blended / n_groups:.1f} objects per blend on average")

## 6.0 Explore Mask Planes

Each `deep_coadd` pixel carries a bitmask of these named planes:

| Mask Plane | Description |
|---|---|
| `NO_DATA` | No data were available for this pixel; common at tract/patch edges, in areas not covered by any input visit, or where all inputs were masked |
| `INTERPOLATED` | The pixel value is the result of interpolating nearby good pixels; generally fine for most analyses. |
| `COSMIC_RAY` | A cosmic ray affected this pixel on at least one input image (and was interpolated over) |
| `SATURATED` | More than 10% of the potential input visits had a saturated pixel at this location |
| `DETECTION_EDGE` | Pixel was too close to the edge of the patch to be considered for detection, due to the finite size of the detection kernel. |
| `CLIPPED` | The region was identified as a probable artifact when comparing multiple single-visit warps, and was excluded from the coadd |
| `REJECTED` | At least one input visit was left out of the coadd for this pixel due to masking. |
| `DETECTED` | Pixel was part of a detected source footprint on the coadd (informational, not a quality flag). |
| `INEXACT_PSF` | The set of visits contributing to this pixel differs from the set contributing to the PSF model for its cell |

In [ ]:
# Ordered so the large planes are drawn first and small ones stay visible on top
plane_colors = {
    "INEXACT_PSF": "#56B4E9",
    "DETECTED": "#009E73",
    "DETECTION_EDGE": "#0072B2",
    "NO_DATA": "#7F7F7F",
    "REJECTED": "#785EF0",
    "CLIPPED": "#E69F00",
    "INTERPOLATED": "#F0E442",
    "COSMIC_RAY": "#D55E00",
    "SATURATED": "#CC79A7",
}
plane_names = list(plane_colors)


In [ ]:
# Checkboxes toggle planes on/off and redraw the image
import ipywidgets as widgets
from IPython.display import display as ipy_display

checkboxes = {
    name: widgets.Checkbox(value=True, description=name, indent=False)
    for name in plane_names
}
output = widgets.Output()

def redraw(*_):
    with output:
        output.clear_output(wait=True)
        ax, extent = plot_coadd(deep_coadd)
        handles = []
        for name, cb in checkboxes.items():
            if cb.value:
                plane = deep_coadd.mask.get(name)
                ax.imshow(
                    np.ma.masked_where(~plane, plane), origin="lower", extent=extent,
                    cmap=ListedColormap([plane_colors[name]]), alpha=0.6,
                )
                handles.append(Patch(facecolor=plane_colors[name], alpha=0.6, label=name))
        if handles:
            ax.legend(handles=handles, title="Mask planes", loc="upper left", bbox_to_anchor=(1.02, 1))
        plt.show()

for cb in checkboxes.values():
    cb.observe(redraw, names="value")

ipy_display(widgets.GridBox(
    list(checkboxes.values()), layout=widgets.Layout(grid_template_columns="repeat(3, 160px)")
))
ipy_display(output)
redraw()

## 8.0 Raise the S/N Threshold

Every catalog comes with a signal-to-noise cut, whether you chose one or not. Drag the slider to raise the minimum PSF-flux S/N: the gray histogram is every source in this patch, the blue one is what survives the cut.

Where does the distribution peak, and why does it turn over at faint magnitudes? How does the faintest surviving magnitude change as you raise the cut?

In [ ]:
# Every object in this patch with its PSF magnitude and S/N (no S/N cut yet)
psf_columns = ["patch", f"{band}_psfFlux", f"{band}_psfFluxErr"]
psf_cat = butler.get("object", tract=dataId["tract"], parameters={"columns": psf_columns})
psf_cat = psf_cat[psf_cat["patch"] == dataId["patch"]]

psf_flux = np.asarray(psf_cat[f"{band}_psfFlux"], dtype=float)
psf_err = np.asarray(psf_cat[f"{band}_psfFluxErr"], dtype=float)
positive = np.isfinite(psf_flux) & np.isfinite(psf_err) & (psf_flux > 0) & (psf_err > 0)
cat = pd.DataFrame({
    "mag": -2.5 * np.log10(psf_flux[positive]) + 31.4,
    "snr": psf_flux[positive] / psf_err[positive],
})
print(f"{positive.sum()} of {len(psf_cat)} objects in this patch have a positive {band}-band PSF flux")

In [ ]:
mag_all = cat["mag"].values
snr_all = cat["snr"].values
# Fixed bins and axis limits so the picture only changes because of the cut
mag_bins = np.arange(np.floor(np.percentile(mag_all, 0.5)), np.ceil(mag_all.max()) + 0.5, 0.5)
counts_all, _ = np.histogram(mag_all, bins=mag_bins)

out_hist = widgets.Output()

def plot_hist(snr_min):
    """Redraw the histogram for a minimum S/N inside the output widget (works with %matplotlib inline)."""
    kept = mag_all[snr_all > snr_min]
    with out_hist:
        out_hist.clear_output(wait=True)
        fig_hist, ax_hist = plt.subplots(figsize=(6, 4))
        ax_hist.hist(mag_all, bins=mag_bins, color="lightgray", label=f"all sources ({len(mag_all)})")
        ax_hist.hist(kept, bins=mag_bins, color="C0", label=rf"$\mathrm{{S/N}} > {snr_min:.0f}$ ({len(kept)})")
        faintest = f"; faintest kept: {kept.max():.1f}" if len(kept) else ""
        ax_hist.set(xlabel=f"${band}$-band PSF magnitude (AB)", ylabel="Number of sources",
                    xlim=(mag_bins[0], mag_bins[-1]), ylim=(0, 1.1 * counts_all.max()),
                    title=f"Magnitude distribution after S/N cut{faintest}")
        ax_hist.legend(loc="upper left")
        plt.show()

snr_slider = widgets.FloatLogSlider(
    value=5, base=10, min=np.log10(5), max=3, step=0.02,
    description="min S/N", readout_format=".0f", continuous_update=False,
    style={"description_width": "60px"}, layout=widgets.Layout(width="500px"),
)
snr_slider.observe(lambda change: plot_hist(change["new"]), names="value")

ipy_display(widgets.VBox([snr_slider, out_hist]))
plot_hist(snr_slider.value)

### Galaxy counts vs. the LSST Science Book

The S/N cut decides where the catalog runs out. How deep *should* the counts go? The LSST Science Book (2009, eq. 3.7) fits the cumulative galaxy counts of the CFHTLS Deep fields with

$$N(<i) = 46 \times 10^{0.31\,(i-25)}\ \text{gal arcmin}^{-2},$$

valid over $20.5 < i < 25.5$. It is the same formula Notebook 2A used for the weak-lensing galaxy density. Here we count the galaxies in this patch and plot them against it.

The choice of galaxies matters, so we take Eli's cuts from his tutorial. Magnitudes come from the **Sérsic** fluxes (`i_sersicFlux`: one Sérsic profile fit to all bands at once, see [tutorial 303.1](https://dp2.lsst.io/tutorials/notebook/303/notebook-303-1.html)), and an object counts as a galaxy if `griz_model_extendedness > 0.2`. That column uses the Sérsic model's flux and size in *griz* to score each object from 0 (compact) to 1 (extended). Objects whose Sérsic fit failed (`sersic_no_data_flag`, `sersic_unknown_flag`) are dropped.

In [ ]:
# Galaxies for the counts: Sersic fluxes and the griz model extendedness
EXT_MIN = 0.2  # griz_model_extendedness above this counts as a galaxy

gal_columns = [
    "patch", "coord_ra", "coord_dec",
    "i_sersicFlux", "i_sersicFluxErr", "sersic_no_data_flag", "sersic_unknown_flag",
    "griz_model_extendedness", "refExtendedness",  # refExtendedness is only for comparison
]
gal = butler.get("object", tract=dataId["tract"], parameters={"columns": gal_columns})
gal = gal[gal["patch"] == dataId["patch"]]

gal_flux = np.asarray(gal["i_sersicFlux"], dtype=float)
gal_snr = gal_flux / np.asarray(gal["i_sersicFluxErr"], dtype=float)
good_fit = (
    np.isfinite(gal_snr) & (gal_flux > 0)
    & ~np.asarray(gal["sersic_no_data_flag"], dtype=bool)
    & ~np.asarray(gal["sersic_unknown_flag"], dtype=bool)
)
model_ext = np.asarray(gal["griz_model_extendedness"], dtype=float)
ref_ext = np.asarray(gal["refExtendedness"], dtype=float)
extended = model_ext > EXT_MIN  # NaN (no classification) counts as not extended
gal_mag = -2.5 * np.log10(np.where(gal_flux > 0, gal_flux, np.nan)) + 31.4

# Area of the patch's inner box, in arcmin^2
skymap = butler.get("skyMap", skymap=SKYMAP)
inner_bbox = skymap[dataId["tract"]].getPatchInfo(dataId["patch"]).inner_bbox
inner_pixels = inner_bbox.getArea() if hasattr(inner_bbox, "getArea") else 3300**2  # 22 x 22 cells of 150 pixels
patch_area = inner_pixels * (PIXEL_SCALE / 60) ** 2
print(f"{(good_fit & extended).sum()} galaxies in {patch_area:.0f} arcmin^2 ({inner_pixels:.0f} pixels)")

`science_book_counts` is the formula. `plot_counts` takes a dictionary of selections (a label and a boolean mask over the catalog) and draws each one's cumulative counts per arcmin², with a $\pm\sqrt{N}$ Poisson band, next to the formula. The lower panel is the ratio of catalog to formula. The shaded range is where the formula was fit.

In [ ]:
def science_book_counts(i_mag):
    """LSST Science Book (2009) eq. 3.7: cumulative galaxy counts N(<i) in arcmin^-2 (fit to 20.5 < i < 25.5)."""
    return 46 * 10 ** (0.31 * (np.asarray(i_mag, dtype=float) - 25))


def plot_counts(selections, mags=gal_mag, area=patch_area):
    """Cumulative counts per arcmin^2 for each selection, against the Science Book fit. Returns the figure."""
    grid = np.arange(18.0, 28.01, 0.1)
    fig_c, (ax_c, ax_r) = plt.subplots(
        2, 1, figsize=(6, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05}
    )
    model = science_book_counts(grid)
    for k, (label, mask) in enumerate(selections.items()):
        sorted_mags = np.sort(mags[mask & np.isfinite(mags)])
        n = np.searchsorted(sorted_mags, grid, side="right")
        density = n / area
        line, = ax_c.plot(grid, np.where(n > 0, density, np.nan), color=f"C{k}", label=label)
        ax_c.fill_between(grid, (n - np.sqrt(n)) / area, (n + np.sqrt(n)) / area, color=f"C{k}", alpha=0.2, lw=0)
        ax_r.plot(grid, np.where(n > 0, density / model, np.nan), color=f"C{k}")
    ax_c.plot(grid, model, "k--", label="LSST Science Book, eq. 3.7")
    for ax_ in (ax_c, ax_r):
        ax_.axvspan(20.5, 25.5, color="gray", alpha=0.12, lw=0)
    ax_c.set(yscale="log", ylabel=r"$N(<i)$ [arcmin$^{-2}$]", ylim=(1e-2, 1e3))
    ax_c.legend(loc="upper left", fontsize=10)
    ax_r.axhline(1, color="k", ls="--", lw=1)
    ax_r.set(xlabel=r"$i$ (Sérsic, AB)", ylabel="catalog / fit", ylim=(0, 2), xlim=(grid[0], grid[-1]))
    return fig_c


selections = {
    "galaxies": good_fit & extended,
    r"galaxies, $\mathrm{S/N} > 20$": good_fit & extended & (gal_snr > 20),
}
fig_counts = plot_counts(selections)
plt.show()

**Where do they agree?** Inside the shaded range, is the catalog within the Poisson band of the fit? Away from the edges of that range, how far off is it, and is that larger than the $\pm\sqrt{N}$ you would expect from one patch? What other source of scatter does a single patch have?

**QUESTION:** Is the turnover in the counts telling you about the universe or about the pipeline? Add your own entry to `selections` and call `plot_counts(selections)` again. Drop the extendedness cut, change `EXT_MIN`, or count only the compact objects (`model_ext <= EXT_MIN`). `refExtendedness` is loaded too (as `ref_ext`), so you can compare with the older PSF-to-cModel flux-ratio classifier (`ref_ext == 1`). What changes, and at which magnitudes? 

- The [DP2 flag guidance](https://dp2.lsst.io/products/flags/flag_recommendations.html) warns that extendedness has not been characterized for purity or completeness. (In HSC, over 20% of stars at $22.5 < i < 25$ were classified as extended; [Huang et al. 2018](https://arxiv.org/abs/1705.01599).)
- The honest way to measure completeness is to inject fake galaxies of known magnitude into the image and count how many come back ([DP1 tutorial 105.4](https://dp1.lsst.io/tutorials/notebook/105/notebook-105-4.html)). 

## 9.0 Overlay Survey Property Maps on the Patch

Now put the maps on the image. `map_on_patch` reads a map at the sky position of every cell in a coarse grid over the patch from section 3.0, and `show_map_on_patch` draws that grid on top of the coadd. Pick a map, a resolution and an opacity, and turn the catalog back on to see where the S/N > 300 sources sit.

Try the PSF FWHM map at native resolution, then at `nside = 512`. The native pixels are about 32 coadd pixels across, so even the best map is blocky next to the image, and at `nside = 512` the whole patch is a handful of pixels. Does the catalog's depth (section 8.0) follow the depth map?

If the widgets don't appear (e.g. in a PDF), call `show_map_on_patch("psf_maglim", 4096)` directly.

In [ ]:
def map_on_patch(key, nside, n=200):
    """A map on an n x n grid of cell centers covering the coadd image, in display units (NaN outside the map)."""
    bbox = deep_coadd.bbox
    x_edges = np.linspace(bbox.x.start - 0.5, bbox.x.stop - 0.5, n + 1)
    y_edges = np.linspace(bbox.y.start - 0.5, bbox.y.stop - 0.5, n + 1)
    X, Y = np.meshgrid(0.5 * (x_edges[1:] + x_edges[:-1]), 0.5 * (y_edges[1:] + y_edges[:-1]))
    sky = deep_coadd.astropy_wcs.pixel_to_world(X, Y)
    return sample(get_map(key, nside), sky.ra.deg, sky.dec.deg) * MAPS[key]["scale"]


def show_map_on_patch(key="psf_maglim", nside=4096, alpha=0.5, catalog=False):
    """The coadd with a survey property map on top and, optionally, the Sérsic S/N > 300 catalog. Returns the axes."""
    ax_, extent_ = plot_coadd(deep_coadd)
    values = map_on_patch(key, nside)
    if np.isfinite(values).any():
        lo, hi = np.nanpercentile(values, [1, 99])
        im = ax_.imshow(
            values, origin="lower", extent=extent_, cmap="viridis", alpha=alpha,
            vmin=lo, vmax=hi, interpolation="nearest",
        )
        ax_.figure.colorbar(im, ax=ax_, shrink=0.8, pad=0.02, label=MAPS[key]["label"])
    if catalog:
        ax_.scatter(objects_xy.x, objects_xy.y, s=60, facecolors="none", edgecolors="tab:orange", linewidths=0.8)
    ax_.set_title(f"{MAPS[key]['label'].split(' [')[0]}, nside = {nside}")
    return ax_

In [ ]:
style = {"description_width": "initial"}
map_dd = widgets.Dropdown(
    options=[(spec["label"], key) for key, spec in MAPS.items()], value="psf_maglim",
    description="Map", style=style,
)
res_dd = widgets.Dropdown(
    options=[("native (6 arcsec)", NSIDE_NATIVE), ("nside 4096 (52 arcsec)", 4096), ("nside 512 (412 arcsec)", 512)],
    value=4096, description="Resolution", style=style,
)
alpha_sl = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05, description="Opacity", continuous_update=False, style=style,
)
cat_cb = widgets.Checkbox(value=False, description="Catalog (Sérsic S/N > 300)", indent=False)
out_map = widgets.Output()
_current = {}

def redraw_map(*_):
    with out_map:
        out_map.clear_output(wait=True)
        if "fig" in _current:
            plt.close(_current["fig"])  # one figure at a time
        with plt.ioff():
            ax_ = show_map_on_patch(map_dd.value, res_dd.value, alpha_sl.value, cat_cb.value)
        _current["fig"] = ax_.figure
        plt.show()

for w in (map_dd, res_dd, alpha_sl, cat_cb):
    w.observe(redraw_map, names="value")

ipy_display(widgets.VBox([widgets.HBox([map_dd, res_dd]), widgets.HBox([alpha_sl, cat_cb]), out_map]))
redraw_map()

**QUESTION:** How sensitive might different cosmological probes be to variations in different survey properties (e.g. PSF size, limiting magnitude, etc.)?

## 10.0 Exercise: Single-Visit vs Coadd Depth

Two numbers describe how deep the data are at this position. The maps give a **PSF magnitude limit**, a depth estimate for the coadd built from its PSF size and noise. The VisitDetector table ([tutorial 201.11](https://dp2.lsst.io/tutorials/notebook/201/notebook-201-11.html)) gives `magLim`, the 5σ point-source limiting magnitude of each single detector image. This exercise puts them side by side and asks whether stacking gains what Week 2 predicted.

**Two predictions from Week 2.**

- **Stacking.** Notebook 2A (Monday's worksheet): a coadd of $N$ equal visits is deeper than one visit by $1.25\,\log_{10} N$, assuming the same sky noise and PSF among the input visits.
- **PSF size.** Notebook 2B (Parts 1 and 2): in sky-noise-dominated images, the depth shifts by $-2.5\,\log_{10}(\text{FWHM}/\text{FWHM}_\text{ref})$. A PSF 10% wider costs about 0.1 mag in PSF photometry.


### Single visits

Every row of the VisitDetector table is one detector image. We keep the ones in our band whose center is within 0.1° of the position (a detector is about 0.23° across, so the position is on or right next to each one), then take medians.

In [ ]:
VISIT_EXPTIME = 30.0  # s

vd_ref = butler.query_datasets("visit_detector_table", instrument="LSSTCam")[0]
vd_columns = ["visitId", "detector", "band", "ra", "dec", "magLim", "psfSigma", "seeing"]
vd = butler.get(vd_ref, parameters={"columns": vd_columns})
vd = vd.to_pandas() if hasattr(vd, "to_pandas") else vd
vd = vd[vd["band"] == band]

separation = SkyCoord(vd["ra"].values * u.deg, vd["dec"].values * u.deg).separation(SkyCoord(ra * u.deg, dec * u.deg))
near_visits = vd[separation < 0.1 * u.deg]

visit_depth = near_visits["magLim"].median()
visit_fwhm = near_visits["psfSigma"].median() * SIGMA_TO_FWHM * PIXEL_SCALE  # same definition as the map
print(f"{len(near_visits)} detector images from {near_visits['visitId'].nunique()} visits in the {band} band")
print(f"median single-visit depth: {visit_depth:.2f} mag; median PSF FWHM: {visit_fwhm:.2f} arcsec")

### The coadd

The same numbers from the maps, as medians over the patch at `nside = 4096`. Dividing the exposure time by the length of a visit gives the number of visits that went in.

In [ ]:
coadd_values = {key: float(np.nanmedian(map_on_patch(key, 4096))) for key in MAPS}
n_visits = coadd_values["exposure_time"] / VISIT_EXPTIME

gain_measured = coadd_values["psf_maglim"] - visit_depth
gain_stacking = 1.25 * np.log10(n_visits)
gain_psf = -2.5 * np.log10(coadd_values["psf_fwhm"] / visit_fwhm)

print(pd.DataFrame(
    {"single visit": [visit_depth, visit_fwhm], "coadd": [coadd_values["psf_maglim"], coadd_values["psf_fwhm"]]},
    index=["depth [mag]", "PSF FWHM [arcsec]"],
).round(2))
print(f"\nvisits in the coadd (exposure time / {VISIT_EXPTIME:.0f} s): {n_visits:.1f}")
print(f"measured gain, coadd - single visit:       {gain_measured:+.2f} mag")
print(f"predicted by stacking, 1.25 log10(N):      {gain_stacking:+.2f} mag")
print(f"predicted by the PSF size change:          {gain_psf:+.2f} mag")
print(f"stacking + PSF size:                       {gain_stacking + gain_psf:+.2f} mag")
print(f"left unexplained:                          {gain_measured - gain_stacking - gain_psf:+.2f} mag")

**Account for the difference.** How close is the prediction? What might cause this prediction to break. You are free to run this notebook again with different patched of sky to see if the expected coadd depth matches the expectation in all scenes.